# 피드백 추천 1~3위 국면모델 구현 및 검증

이 노트북은 다음 세 개선안을 **실제 코드로 재현**하고 현재 `VKOSPI 동적 전략`과 비교합니다.

1. **CJM → LightGBM → Probability Calibration**
2. **TVTP-HMM**
3. **CJM 단독**

각 모델은 1·3·6개월 Risk-Off 확률을 별도로 만들며, Brier·LogLoss·ECE·국면전환 Recall과 CAGR·Sharpe·MDD를 함께 평가합니다. 연구용 시뮬레이션이며 투자 조언이 아닙니다.


## 1. Colab 런타임과 패키지

In [ ]:
import sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([
        sys.executable, "-m", "pip", "install", "-q",
        "lightgbm==4.6.0", "jumpmodels", "statsmodels==0.14.6",
        "openpyxl", "numpy", "pandas", "matplotlib", "scikit-learn",
    ], check=True)

import json, shutil, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

print("Colab:", IN_COLAB, "| Python:", sys.version.split()[0])


## 2. 실행 번들 불러오기

Colab에서는 함께 제공된 `top3_regime_models_colab_bundle.zip`을 업로드합니다. ZIP은 경로 순회를 검사한 뒤 `/content/RegimeDecisionTest`에 해제합니다. 로컬에서는 노트북을 프로젝트 루트에서 실행하면 됩니다.


In [ ]:
def safe_extract(path: Path, destination: Path) -> None:
    destination = destination.resolve()
    with zipfile.ZipFile(path) as archive:
        for member in archive.infolist():
            target = (destination / member.filename).resolve()
            if target != destination and destination not in target.parents:
                raise ValueError(f"Unsafe ZIP member: {member.filename}")
        archive.extractall(destination)

if IN_COLAB:
    from google.colab import files
    uploaded = files.upload()
    bundle = next((Path(name) for name in uploaded if name.endswith(".zip")), None)
    if bundle is None:
        raise FileNotFoundError("top3_regime_models_colab_bundle.zip을 업로드하세요.")
    safe_extract(bundle, Path("/content"))
    PROJECT_ROOT = Path("/content/RegimeDecisionTest")
else:
    PROJECT_ROOT = Path.cwd().resolve()

required = [
    "strategies/stage07_regime_models/top3_regime_model_experiment.py", "strategies/core/regime_research.py",
    "raw_data/compass.db", "cache/market_daily.csv",
    "cache/stress_monthly.csv", "results/vkospi_dynamic_reconciled_monthly.csv",
]
missing = [name for name in required if not (PROJECT_ROOT / name).exists()]
if missing:
    raise FileNotFoundError(missing)
print("PROJECT_ROOT:", PROJECT_ROOT)


## 3. 전체 워크포워드 실행

Colab 기본값은 `True`입니다. 약 수 분 동안 매월 다음 작업을 반복합니다.

- CJM: 과거 144개월 이내 자료만 robust scaling → `JumpModel(cont=True)` → 온라인 확률
- TVTP-HMM: 과거 수익률의 2상태 Gaussian emission + `exog_tvtp`
- LightGBM: horizon별로 **결과가 이미 알려진 라벨만** 학습 → 최근 15~24개월 Platt calibration
- 오버레이 선택: 2017-12까지만 사용, 2018-01 이후 잠금


In [ ]:
RUN_FULL_EXPERIMENT = True if IN_COLAB else False

if RUN_FULL_EXPERIMENT:
    completed = subprocess.run(
        [
            sys.executable,
            "-u",
            "-m",
            "strategies.stage07_regime_models.top3_regime_model_experiment",
        ],
        cwd=PROJECT_ROOT, text=True, capture_output=True, check=True,
    )
    print(completed.stdout[-12000:])
else:
    print("번들의 사전 계산 결과를 사용합니다.")


## 4. 코드 구조와 입력변수

In [ ]:
sys.path.insert(0, str(PROJECT_ROOT))
import strategies.stage07_regime_models.top3_regime_model_experiment as experiment
features, risk_off_label, asset_returns, baseline = experiment.build_master_features()
print("특성 패널:", features.shape, features.index.min(), "→", features.index.max())
display(pd.DataFrame({
    "CJM 입력": pd.Series(experiment.CJM_CANDIDATES),
    "TVTP 전이식": pd.Series(experiment.TVTP_CANDIDATES),
    "LightGBM 우선 입력": pd.Series(experiment.LGBM_PREFERRED),
}))
display(features.tail(3))


### 입력변수 설계

- **Level:** GDP/수출/BSI, VIX, BAA spread, NFCI/STLFSI, VKOSPI, 시장 수익·변동성
- **Momentum:** 1개월·3개월 변화, 자산 3/6개월 누적수익
- **Z-score:** 60개월 스트레스 표준화, VKOSPI 63/252일 표준화
- **Acceleration:** 최근 변화량의 재변화
- **CJM 상태:** 현재 Risk-Off 확률, 1/3/6개월 전파확률, 확률 변화, 지속기간, 최근 전환 횟수

모든 행은 해당 월말까지 알려진 값입니다. Risk-Off 정답은 그 뒤 KODEX200 월간 오픈-투-오픈 수익이 음수인지로 정의합니다.


## 5. 세 모델의 실제 구현 방식

### CJM

공식 `jumpmodels.jump.JumpModel(cont=True, grid_size=0.10)`을 매월 다시 적합합니다. 수익률이 낮은 상태를 Risk-Off로 정렬하고 `predict_proba_online()`의 마지막 확률을 사용합니다. 전이행렬을 1·3·6번 곱해 미래 확률을 만듭니다.

### TVTP-HMM

`statsmodels.MarkovRegression(..., exog_tvtp=X)`의 전이확률은 고정 상수가 아니라 신용·금융여건·VIX/VKOSPI의 함수입니다. 전이식에는 한 달 전 관측치를 넣어 현재 상태 설명에 현재 정보를 역사용하지 않습니다.

### CJM + LightGBM

각 horizon을 별도 이진분류로 적합합니다. 작은 월간 표본을 고려해 깊이 3, 잎 7개로 제한하고, 마지막 검증구간 raw probability의 logit을 LogisticRegression에 넣어 Platt calibration합니다.


## 6. 미래 정보 차단 감사

In [ ]:
RESULTS = PROJECT_ROOT / "results"
audit = pd.read_csv(RESULTS / "top3_regime_model_lgbm_audit.csv")
signal = pd.PeriodIndex(audit["signal_month"], freq="M")
max_label = pd.PeriodIndex(audit["max_label_month"], freq="M")
fit_end = pd.PeriodIndex(audit["fit_end_month"], freq="M")
assert (max_label <= signal).all()
assert (fit_end < signal).all()
display(audit.tail())
print("통과: 모든 LightGBM 학습 라벨의 실현월 ≤ 신호월, 학습 특성월 < 신호월")


## 7. 확률 품질 비교

In [ ]:
prediction_metrics = pd.read_csv(RESULTS / "top3_regime_model_prediction_metrics.csv")
locked_probability = prediction_metrics.query("Period == 'locked_2018_2026'")
display(locked_probability.style.format({
    "Brier":"{:.3f}", "LogLoss":"{:.3f}", "ECE5":"{:.3f}",
    "AUC":"{:.3f}", "BalancedAccuracy":"{:.3f}", "TransitionRecall":"{:.1%}",
}))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, horizon in zip(axes, [1, 3, 6]):
    view = locked_probability.query("Horizon == @horizon").sort_values("Brier")
    ax.bar(view["Model"], view["Brier"], color=["#007a5e" if "LightGBM" in x else "#87928d" for x in view["Model"]])
    ax.set_title(f"{horizon}M Brier ↓")
    ax.tick_params(axis="x", rotation=45)
plt.tight_layout(); plt.show()


## 8. 2017년까지 선택된 포트폴리오 규칙

In [ ]:
calibration = pd.read_csv(RESULTS / "top3_regime_model_calibration.csv")
winners = calibration.loc[calibration["Selected"].fillna(False)]
display(winners[[
    "Model","threshold","max_shift","bond_share","horizon_weights",
    "CAGR","Sharpe","MDD","Calmar","AvgDefensiveShift","ActiveMonths",
]].style.format({"CAGR":"{:.2%}","MDD":"{:.2%}","Sharpe":"{:.3f}","Calmar":"{:.3f}"}))


### 방어 혼합 코드의 의미

확률 가중치는 `p = w1·p1M + w3·p3M + w6·p6M`입니다. `p`가 문턱보다 높을 때만 `shift = max_shift × (p-threshold)/(1-threshold)`를 활성화합니다. 기존 전략의 일부를 BOND/GLD 수익으로 대체하고 왕복 거래비용과 GLD 환전비용을 추가 차감합니다. 도전 모델끼리는 완전히 같은 규칙·그리드를 사용합니다.


## 9. 전체기간·잠금기간 성과

In [ ]:
comparison = pd.read_csv(RESULTS / "top3_regime_model_comparison.csv")
display(comparison.style.format({
    "CAGR":"{:.2%}","Volatility":"{:.2%}","Sharpe":"{:.3f}",
    "MDD":"{:.2%}","Calmar":"{:.3f}","AvgTurnover":"{:.3f}",
}))

paths = {"Existing_VKOSPI_Dynamic": pd.read_csv(RESULTS/"vkospi_dynamic_reconciled_monthly.csv", index_col="month")}
for name, slug in [("CJM","cjm"),("TVTP-HMM","tvtp_hmm"),("CJM+LightGBM","cjm_plus_lightgbm")]:
    paths[name] = pd.read_csv(RESULTS/f"top3_regime_model_backtest_{slug}.csv", index_col="month")

fig, axes = plt.subplots(2,1,figsize=(14,9),sharex=True,gridspec_kw={"height_ratios":[2,1]})
colors={"Existing_VKOSPI_Dynamic":"#87928d","CJM":"#377dff","TVTP-HMM":"#e07a3f","CJM+LightGBM":"#007a5e"}
for name, path in paths.items():
    axes[0].plot(path.index, path["nav"], label=name, color=colors[name], lw=2)
    axes[1].plot(path.index, 100*path["drawdown"], label=name, color=colors[name], lw=1.6)
axes[0].set_title("Cumulative NAV"); axes[0].legend(); axes[0].grid(alpha=.2)
axes[1].set_title("Drawdown (%)"); axes[1].grid(alpha=.2)
axes[1].set_xticks(range(0,len(paths["CJM"]),24), paths["CJM"].index[::24], rotation=45)
plt.tight_layout(); plt.show()


## 10. 잠금 부트스트랩과 최종 판단

In [ ]:
report = json.loads((RESULTS/"top3_regime_model_validation.json").read_text(encoding="utf-8"))
rows=[]
for model, detail in report["validation"].items():
    rows.append({"Model":model, **detail["locked_deltas"], **detail["bootstrap"], "passes_all_three":detail["passes_all_three"]})
validation_table=pd.DataFrame(rows)
display(validation_table.style.format({
    "CAGR":"{:+.2%}","Sharpe":"{:+.3f}","MDD":"{:+.2%}","Calmar":"{:+.3f}",
    "probability_cagr_improves":"{:.1%}","probability_sharpe_improves":"{:.1%}",
    "probability_mdd_improves":"{:.1%}","probability_all_three_improve":"{:.1%}",
}))

promoted=validation_table.loc[validation_table["passes_all_three"],"Model"].tolist()
print("세 지표 동시 개선 모델:", promoted if promoted else "없음 — 기존 전략 유지가 보수적 결론")


## 11. LightGBM 변수 중요도

In [ ]:
importance=pd.read_csv(RESULTS/"top3_regime_model_feature_importance.csv").head(20)
display(importance)
plt.figure(figsize=(10,6)); plt.barh(importance["feature"][::-1],importance["mean_gain_share"][::-1],color="#007a5e")
plt.title("Mean LightGBM gain share"); plt.tight_layout(); plt.show()


## 12. 한계와 운영 전 체크리스트

- 월간 표본이 작아 복잡한 상호작용은 불안정할 수 있습니다.
- CJM/TVTP의 잠재상태는 관측 가능한 경제적 진실이 아니라 모델 근사입니다.
- Brier가 좋아도 포트폴리오 성과가 좋아진다는 보장은 없습니다.
- 세금, 상품 추적오차, 실제 체결 슬리피지, 레버리지 한도는 추가 반영해야 합니다.
- 운영 전에는 매월 데이터 가용일, 수정치(vintage), 모델 수렴, 확률 calibration drift를 모니터링해야 합니다.


## 13. 결과 ZIP 저장

In [ ]:
output = Path("/content/top3_regime_model_results.zip") if IN_COLAB else PROJECT_ROOT/"top3_regime_model_results.zip"
with zipfile.ZipFile(output,"w",compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(RESULTS.glob("top3_regime_model_*")):
        archive.write(path,arcname=path.name)
print("저장:",output)
if IN_COLAB:
    from google.colab import files
    files.download(str(output))
